# Cross-Industry Accelerator — Create Data Agents
### Auto-Create Fabric IQ QA and Operations Agents

This notebook creates **two agents** backed by the ontology and data sources:

| Agent | Purpose | Power |
|---|---|---|
| **QA Agent** | Answer ad-hoc data questions in natural language | Queries Lakehouse + Eventhouse via ontology |
| **Ops Agent** | Monitor operations and surface alerts/anomalies | Real-time event analysis via ontology |

**What it does:**
1. Reads industry configuration from `00_Industry_Config`
2. Resolves the ontology item ID from the workspace
3. Creates the QA Agent item linked to the ontology
4. Creates the Ops Agent item linked to the ontology
5. Configures each agent with industry-appropriate instructions

> **Prerequisites:**
> 1. Run `05_Create_Ontology.ipynb` to create the ontology first
> 2. Run `04_Create_Semantic_Model.ipynb` to create the semantic model
> 3. Data must be loaded (Lakehouse + Eventhouse) for agents to query

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════
%run ./00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RESOLVE WORKSPACE ITEMS
# ════════════════════════════════════════════════════════════════════════

import sempy.fabric as fabric
import json, requests

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
items_df = fabric.list_items()

# Resolve Ontology item ID
ont_matches = items_df[(items_df["Type"] == "Ontology") & (items_df["Display Name"] == ONTOLOGY_NAME)]
if ont_matches.empty:
    print(f"⚠ Ontology '{ONTOLOGY_NAME}' not found. Run 05_Create_Ontology first.")
    print(f"Available ontologies:")
    print(items_df[items_df["Type"] == "Ontology"][["Display Name", "Id"]].to_string(index=False))
    ontology_item_id = None
else:
    ontology_item_id = str(ont_matches.iloc[0].Id)
    print(f"✓ Ontology: {ONTOLOGY_NAME} → {ontology_item_id}")

# Resolve Lakehouse item ID
lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
lakehouse_item_id = str(lh_matches.iloc[0].Id) if not lh_matches.empty else None
print(f"✓ Lakehouse: {LAKEHOUSE_NAME} → {lakehouse_item_id}")

# Resolve Eventhouse item ID
eventhouse_item_id = None
if EVENTHOUSE_NAME:
    eh_matches = items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == EVENTHOUSE_NAME)]
    if not eh_matches.empty:
        eventhouse_item_id = str(eh_matches.iloc[0].Id)
        print(f"✓ Eventhouse: {EVENTHOUSE_NAME} → {eventhouse_item_id}")

print(f"\n  QA Agent name:  {DATA_AGENT_NAME}")
print(f"  Ops Agent name: {OPS_AGENT_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# AGENT CREATION HELPER
# ════════════════════════════════════════════════════════════════════════

FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"

def create_fabric_agent(agent_name, agent_description, ontology_id, system_instructions):
    """Create a Fabric IQ Agent item linked to an ontology."""
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    # Agent definition payload with ontology binding
    agent_definition = {
        "ontologyId": ontology_id,
        "systemInstructions": system_instructions,
    }

    payload = {
        "displayName": agent_name,
        "description": agent_description,
        "type": "DataAgent",
        "definition": {
            "parts": [
                {
                    "path": "agent-definition.json",
                    "payload": json.dumps(agent_definition),
                    "payloadType": "InlineBase64"
                }
            ]
        }
    }

    # Base64-encode the payload
    import base64
    definition_json = json.dumps(agent_definition)
    payload["definition"]["parts"][0]["payload"] = base64.b64encode(
        definition_json.encode("utf-8")
    ).decode("utf-8")

    resp = requests.post(url, headers=headers, json=payload)
    return resp

print("✅ Agent creation helper loaded.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE QA AGENT (Data Questions)
# ════════════════════════════════════════════════════════════════════════

if not ontology_item_id:
    print("⚠ Cannot create agents — ontology not found. Run 05_Create_Ontology first.")
else:
    qa_instructions = (
        f"You are a data analyst agent for the {INDUSTRY} industry. "
        f"Answer questions about {INDUSTRY} data using the linked ontology. "
        f"Query dimension tables for reference data and fact tables for metrics. "
        f"When answering, cite specific data points and provide context. "
        f"If data is insufficient, explain what additional data would be needed."
    )

    print(f"Creating QA Agent: {DATA_AGENT_NAME}...")
    qa_resp = create_fabric_agent(
        agent_name=DATA_AGENT_NAME,
        agent_description=f"QA Data Agent for {INDUSTRY} — answers ad-hoc questions via ontology",
        ontology_id=ontology_item_id,
        system_instructions=qa_instructions,
    )

    if qa_resp.status_code in (200, 201, 202):
        print(f"✅ QA Agent created: {DATA_AGENT_NAME}")
        qa_json = qa_resp.json()
        if "id" in qa_json:
            print(f"   Item ID: {qa_json['id']}")
    else:
        print(f"⚠ QA Agent creation returned status {qa_resp.status_code}")
        print(f"   Response: {qa_resp.text}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE OPS AGENT (Operations & Alerting)
# ════════════════════════════════════════════════════════════════════════

if ontology_item_id:
    ops_instructions = (
        f"You are an operations monitoring agent for the {INDUSTRY} industry. "
        f"Analyze real-time event streams and fact tables to detect anomalies, "
        f"surface operational alerts, and provide trend analysis. "
        f"Focus on KPIs, SLA compliance, quality metrics, and risk indicators. "
        f"When reporting issues, provide severity (Critical/High/Medium/Low) and recommended actions."
    )

    print(f"Creating Ops Agent: {OPS_AGENT_NAME}...")
    ops_resp = create_fabric_agent(
        agent_name=OPS_AGENT_NAME,
        agent_description=f"Operations Agent for {INDUSTRY} — monitors events and surfaces alerts",
        ontology_id=ontology_item_id,
        system_instructions=ops_instructions,
    )

    if ops_resp.status_code in (200, 201, 202):
        print(f"✅ Ops Agent created: {OPS_AGENT_NAME}")
        ops_json = ops_resp.json()
        if "id" in ops_json:
            print(f"   Item ID: {ops_json['id']}")
    else:
        print(f"⚠ Ops Agent creation returned status {ops_resp.status_code}")
        print(f"   Response: {ops_resp.text}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# AGENT CREATION SUMMARY
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*60}")
print(f"DATA AGENT SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  Ontology:       {ONTOLOGY_NAME}")
print(f"  Lakehouse:      {LAKEHOUSE_NAME}")
print(f"  Eventhouse:     {EVENTHOUSE_NAME or 'N/A'}")
print(f"")
print(f"  QA Agent:       {DATA_AGENT_NAME}")
print(f"    → Answers ad-hoc data questions about {INDUSTRY} data")
print(f"    → Queries: dimensions, facts, aggregates")
print(f"")
print(f"  Ops Agent:      {OPS_AGENT_NAME}")
print(f"    → Monitors real-time events and operational metrics")
print(f"    → Surfaces: anomalies, SLA breaches, quality alerts")
print(f"")
print(f"✅ Agent creation complete.")
print(f"   Next: Run 07_Create_Dashboards.ipynb to create real-time and Power BI dashboards.")